# Quito by Time of Day

Four one-hour assignments, compared. The road model does the morning peak alone.

Reads what `TAZ_OD_demo.ipynb` exported and rebuilds none of it:

| needs | from |
|---|---|
| 761 zones | `outputs/taz_od_exports/zones.gpkg` |
| O-D matrix | `outputs/taz_od_exports/od_matrix.omx` |
| network and connectors | `data/road_model/` |

All committed, so `data/gov_data/` is not needed.

In [ ]:
import os
import shutil

import folium
import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

_cwd = os.getcwd()
BASE_DIR = _cwd if os.path.isdir(os.path.join(_cwd, "outputs")) else os.path.dirname(_cwd)
EXPORTS = os.path.join(BASE_DIR, "outputs", "taz_od_exports")
MAPS_DIR = os.path.join(BASE_DIR, "outputs", "time_of_day_maps")
os.makedirs(MAPS_DIR, exist_ok=True)
for _stale in os.listdir(MAPS_DIR):
    _path = os.path.join(MAPS_DIR, _stale)
    if os.path.isfile(_path):
        os.remove(_path)

SOURCE_MODEL = os.path.join(BASE_DIR, "data", "road_model")
WORK_MODEL = os.environ.get("TOD_MODEL_DIR", os.path.join(BASE_DIR, "data", "time_of_day_model"))
METRE_CRS = "EPSG:32717"

for _needed in (os.path.join(EXPORTS, "zones.gpkg"),
                os.path.join(EXPORTS, "od_matrix.omx"),
                os.path.join(SOURCE_MODEL, "project_database.sqlite")):
    if not os.path.exists(_needed):
        raise FileNotFoundError(f"Missing input: {_needed}\n"
                                f"Run TAZ_OD_demo.ipynb first - its last section writes these.")

print(f"exports: {EXPORTS}\nmodel:   {WORK_MODEL}\nmaps:    {MAPS_DIR}")

## A1. The matrix

125,329 vehicle trips, and **one-way**: productions from population, attractions from
non-residential floor area. Home to activity, not an average hour.

Symmetric demand would show asymmetry near 0 and transpose correlation near 1.

In [ ]:
from aequilibrae.matrix import AequilibraeMatrix                       # noqa: E402

zones = gpd.read_file(os.path.join(EXPORTS, "zones.gpkg"))
print(f"{len(zones):,} zones, {zones['pop'].sum():,.0f} people, "
      f"{zones['activity_m2'].sum() / 1e6:,.1f} km2 of non-residential floor area")

_read = AequilibraeMatrix()
_read.create_from_omx(os.path.join(EXPORTS, "od_matrix.omx"),
                      os.path.join(WORK_MODEL + "_tmp.aem"), memory_only=True)
HOME_TO_ACTIVITY = np.array(_read.matrix["car"], dtype=float)
ZONE_IDS = np.array(_read.index[:], dtype=np.int64)
_read.close()
if os.path.exists(WORK_MODEL + "_tmp.aem"):
    os.remove(WORK_MODEL + "_tmp.aem")

_off = HOME_TO_ACTIVITY - np.diag(np.diag(HOME_TO_ACTIVITY))
print(f"\nmatrix {HOME_TO_ACTIVITY.shape}, {HOME_TO_ACTIVITY.sum():,.0f} vehicle trips")
print(f"  intrazonal: {100 * np.trace(HOME_TO_ACTIVITY) / HOME_TO_ACTIVITY.sum():.1f}%")
print(f"  directional asymmetry |T-T'|/T: "
      f"{100 * np.abs(_off - _off.T).sum() / _off.sum():.0f}%")
print(f"  correlation of T with its transpose: "
      f"{np.corrcoef(_off.ravel(), _off.T.ravel())[0, 1]:.3f}")
_rows, _cols = _off.sum(1), _off.sum(0)
print(f"  zones sending more than 1.5x what they receive: "
      f"{(_rows > 1.5 * np.maximum(_cols, 1e-9)).sum()}")
print(f"  zones receiving more than 1.5x what they send:  "
      f"{(_cols > 1.5 * np.maximum(_rows, 1e-9)).sum()}")

## A2. The four hours

**One hour each**, because capacity is vehicles per hour and BPR reads an hourly V/C.

The matrix is directional, so an hour is a transform of it plus a scale — not a second gravity
model:

| hour | matrix | scale |
|---|---|---|
| morning peak 07:00–08:00 | `T` | 1.00 |
| evening peak 17:00–18:00 | `T'` | 1.00 |
| midday 12:00–13:00 | `(T + T')/2` | 0.48 |
| night 22:00–23:00 | `(T + T')/2` | 0.17 |

The morning peak is 1.00 by construction: it is the matrix validated against the AMT corridor
counts. The evening is set equal to it — the same journeys reversed, and though the evening
carries more traffic overall it spreads over more hours, so its busiest hour lands close.
Midday and night are the shape of an ordinary urban profile.

**None of the three is measured.** One AMT count station with an hourly profile would replace
them. Both peak hours sit inside the *pico y placa* windows, 07:00–09:30 and 16:00–19:30.

**Turning the matrix round is a simplification, forced by the data we have.** Combining `T` and
`T'` is the standard way to split a day — practice writes it `f·T + (1-f)·T'` and takes `f` from
a household travel survey, typically nearer 85/15 in a commute peak than the 100/0 used here.
Quito has no such survey, so this notebook takes the extreme: no counter-flow, no reverse
commutes, and one matrix standing in for every trip purpose. The peaks come out more
one-directional than they really are.

In [ ]:
# Demand in each hour, relative to the morning peak. Assumed - see the note above.
PERIOD_SCALE = {"morning peak": 1.00, "evening peak": 1.00, "midday": 0.48, "night": 0.17}

DIRECTION = {"morning peak": "forward", "evening peak": "reverse",
             "midday": "both", "night": "both"}
# One hour each, and each one inside the period it stands for.
HOURS = {"morning peak": "07:00-08:00", "evening peak": "17:00-18:00",
         "midday": "12:00-13:00", "night": "22:00-23:00"}
PERIODS = list(PERIOD_SCALE)


def period_matrix(name):
    """The demand for one period: the base matrix turned round if need be, then scaled."""
    how = DIRECTION[name]
    if how == "forward":
        base = HOME_TO_ACTIVITY
    elif how == "reverse":
        base = HOME_TO_ACTIVITY.T
    else:
        base = (HOME_TO_ACTIVITY + HOME_TO_ACTIVITY.T) / 2
    return base * PERIOD_SCALE[name]


print(f"  {'period':<15}{'hour':<14}{'matrix':<10}{'scale':>7}{'trips':>12}")
for _p in PERIODS:
    print(f"  {_p:<15}{HOURS[_p]:<14}{DIRECTION[_p]:<10}"
          f"{PERIOD_SCALE[_p]:>7.2f}{period_matrix(_p).sum():>12,.0f}")

## A3. The road model

A copy, so four runs writing results leave `data/road_model/` untouched.

In [ ]:
import logging                                                        # noqa: E402

from aequilibrae import Project                                       # noqa: E402
from aequilibrae.paths import TrafficAssignment, TrafficClass         # noqa: E402


def _release_log_files(folder):
    """AequilibraE keeps a log file open inside the project; Windows will not delete it."""
    target = os.path.abspath(folder)
    for logger in [logging.getLogger()] + [logging.getLogger(n) for n
                                           in list(logging.Logger.manager.loggerDict)]:
        for handler in list(getattr(logger, "handlers", [])):
            written = getattr(handler, "baseFilename", None)
            if written and os.path.abspath(written).startswith(target):
                handler.close()
                logger.removeHandler(handler)


def open_working_copy():
    """Copy the saved model and open the copy."""
    if os.path.isdir(WORK_MODEL):
        _release_log_files(WORK_MODEL)
        try:
            shutil.rmtree(WORK_MODEL)
        except PermissionError as broken:
            raise PermissionError(
                f"{WORK_MODEL} is held open by another process, so it cannot be replaced. "
                f"Close any notebook using it, or set TOD_MODEL_DIR to another path.") from broken
    shutil.copytree(SOURCE_MODEL, WORK_MODEL,
                    ignore=shutil.ignore_patterns("*.log", "osm_data"))
    project = Project()
    project.open(WORK_MODEL)
    return project


model = open_working_copy()
with model.db_connection as _conn:
    _links = pd.read_sql("SELECT count(*) n FROM links", _conn).n[0]
    _zones = pd.read_sql("SELECT count(*) n FROM zones", _conn).n[0]
print(f"{_links:,} links, {_zones:,} zones")

model.network.build_graphs(modes=["c"])
car_graph = model.network.graphs["c"]
car_graph.prepare_graph(ZONE_IDS)
car_graph.set_graph("travel_time")
car_graph.set_skimming(["travel_time"])
car_graph.set_blocked_centroid_flows(True)
print(f"graph ready: {car_graph.compact_num_links:,} compact links, "
      f"{len(ZONE_IDS):,} centroids")

## B1. One assignment per hour

Same solver and volume-delay function as the road notebook, so the hours are comparable with it
and with each other. Each hour's flows go to its own table through `save_results`.

**BPR** turns flow and capacity into travel time, with `alpha = 0.15` and `beta = 4.0` — the
values it is almost always published with, carried over unchanged. There is no observed
speed-flow data for Quito's streets.

In [ ]:
BPR_ALPHA, BPR_BETA = 0.15, 4.0

RGAP_TARGET, MAX_ITER = 0.001, 200


def as_matrix(array, name="car"):
    """Wrap a zone-by-zone array as the demand matrix the solver reads."""
    holder = AequilibraeMatrix()
    holder.create_empty(zones=array.shape[0], matrix_names=[name], memory_only=True)
    holder.index[:] = ZONE_IDS
    holder.matrices[:, :, 0] = array
    holder.computational_view([name])
    return holder


def solve(period):
    """One user-equilibrium assignment, saved under the period's name."""
    job = TrafficAssignment()
    job.add_class(TrafficClass(name="car", graph=car_graph,
                               matrix=as_matrix(period_matrix(period))))
    job.set_vdf("BPR")
    job.set_vdf_parameters({"alpha": BPR_ALPHA, "beta": BPR_BETA})
    job.set_capacity_field("capacity")
    job.set_time_field("travel_time")
    job.set_algorithm("bfw")
    job.max_iter = MAX_ITER
    job.rgap_target = RGAP_TARGET
    job.execute()
    table = period.replace(" ", "_")
    job.save_results(table_name=table)
    print(f"{period:<15} gap {job.assignment.rgap:.2e} after {job.assignment.iter} iterations"
          + ("" if job.assignment.rgap <= RGAP_TARGET else "   ** did not reach the target"))
    return job.results()


results = {}
for _p in PERIODS:
    results[_p] = solve(_p)
print(f"\nresults saved in {os.path.join(WORK_MODEL, 'results_database.sqlite')}")

## B2. How the hours differ

V/C is per direction, and a street is filed under its busier one.

In [ ]:
CAR_AB, CAR_BA = "car_ab", "car_ba"

with model.db_connection as _conn:
    links = pd.read_sql(
        "SELECT link_id, name, link_type, direction, distance, capacity_ab, capacity_ba, "
        "travel_time_ab, travel_time_ba, lanes_ab, speed_ab FROM links", _conn)


def loaded_streets(frame):
    """Link results joined to the network, with V/C per direction."""
    out = frame.rename_axis("link_id").reset_index().merge(links, on="link_id", how="left")
    for side, cap in (("ab", "capacity_ab"), ("ba", "capacity_ba")):
        out[f"voc_{side}"] = out[f"car_{side}"].fillna(0) / out[cap].replace(0, np.nan)
    out["voc"] = out[["voc_ab", "voc_ba"]].max(axis=1)
    out["flow"] = out[CAR_AB].fillna(0) + out[CAR_BA].fillna(0)
    out = out[out["link_type"] != "centroid_connector"]
    return out[out["flow"] > 0]


def summary(period):
    """One row of the comparison table."""
    f = loaded_streets(results[period])
    veh_h = float((f[CAR_AB].fillna(0) * f["Congested_Time_AB"].fillna(0)
                   + f[CAR_BA].fillna(0) * f["Congested_Time_BA"].fillna(0)).sum() / 60)
    free_h = float((f[CAR_AB].fillna(0) * f["travel_time_ab"].fillna(0)
                    + f[CAR_BA].fillna(0) * f["travel_time_ba"].fillna(0)).sum() / 60)
    over = f[f["voc"] >= 1.0]
    return {"trips": period_matrix(period).sum(), "loaded links": len(f),
            "veh-km": (f["flow"] * f["distance"] / 1000).sum() / 1000,
            "veh-hours": veh_h, "delay %": 100 * (veh_h / free_h - 1),
            "over capacity km": over["distance"].sum() / 1000,
            "worst V/C": f["voc"].max()}


table = pd.DataFrame({p: summary(p) for p in PERIODS}).T
print(table.to_string(float_format=lambda v: f"{v:,.1f}"))

print("\nthe busiest street in each period, by the flow in one direction:")
for _p in PERIODS:
    f = loaded_streets(results[_p])
    named = f[f["name"].notna() & (f["name"].astype(str) != "")]
    top = named.assign(peak=named[[CAR_AB, CAR_BA]].max(axis=1)).nlargest(1, "peak").iloc[0]
    print(f"  {_p:<15}{str(top['name'])[:34]:<36}{top['peak']:>8,.0f} veh/h  "
          f"V/C {top['voc']:.2f}")

### Does the peak turn round?

The two peaks should load **different sides of the same roads**.

Only two-way streets can show it. A one-way link has bias +1 in every hour whatever the demand
does, and is 36% of the loaded network — pooling them hides the reversal.

In [ ]:
_am = loaded_streets(results["morning peak"])[["link_id", CAR_AB, CAR_BA, "name",
                                              "distance", "direction"]]
_pm = loaded_streets(results["evening peak"])[["link_id", CAR_AB, CAR_BA]]
_both = _am.merge(_pm, on="link_id", suffixes=("_am", "_pm"))


def bias(frame, suffix):
    """How lopsided a link's two directions are: +1 all one way, -1 all the other, 0 balanced."""
    ab = frame[f"{CAR_AB}_{suffix}"].fillna(0)
    ba = frame[f"{CAR_BA}_{suffix}"].fillna(0)
    return (ab - ba) / (ab + ba).replace(0, np.nan)


_both["am_bias"] = bias(_both, "am")
_both["pm_bias"] = bias(_both, "pm")
_both = _both.dropna(subset=["am_bias", "pm_bias"])
_two_way = _both[_both["direction"] == 0]
_one_way = _both[_both["direction"] != 0]
print(f"loaded in both peaks: {len(_both):,} streets "
      f"({len(_two_way):,} two-way, {len(_one_way):,} one-way)")

_pooled = np.corrcoef(_both["am_bias"], _both["pm_bias"])[0, 1]
_real = np.corrcoef(_two_way["am_bias"], _two_way["pm_bias"])[0, 1]
print(f"\ncorrelation of directional bias, morning against evening:")
print(f"  two-way streets only: {_real:+.3f}   <- the figure that means something")
print(f"  every street pooled:  {_pooled:+.3f}   <- one-way links sit at +1 in both peaks")
print("  -1 would mean every street reverses completely, +1 that nothing changes")

_flipped = _two_way[(_two_way["am_bias"] * _two_way["pm_bias"] < 0)
                    & (_two_way["am_bias"].abs() > 0.2)]
print(f"\ntwo-way streets whose busier direction reverses: {len(_flipped):,} of "
      f"{len(_two_way):,} ({100 * len(_flipped) / len(_two_way):.0f}%), "
      f"{_flipped['distance'].sum() / 1000:,.0f} km")

## B3. The hours on a map

One layer per hour, V/C of the busier direction, **mutually exclusive** — choosing one turns the
others off. Through-traffic classes only.

In [ ]:
import shapely                                                        # noqa: E402

# Four periods means four copies of every drawn line, so this map is stricter than the road
# notebook's: tertiary streets are left out. They are 8,272 of the 20,349 links that would
# otherwise be drawn, and at city zoom they read as fill rather than structure.
MAP_ROAD_TYPES = ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link",
                  "secondary", "secondary_link"]
# Bands, colours and labels exactly as the road notebook draws them, so the two congestion maps
# can be read side by side without relearning the palette.
VOC_BANDS = [(0.00, 0.50, "#2c7bb6", "under 50% - free flowing"),
             (0.50, 0.85, "#8cc4de", "50-85% - busy"),
             (0.85, 1.00, "#fdae61", "85-100% - at capacity"),
             (1.00, 1.50, "#d7191c", "100-150% - over capacity"),
             (1.50, 1e9, "#7d0018", "over 150% - far beyond capacity")]
LINE_WEIGHT = {"motorway": 3.6, "trunk": 3.2, "primary": 2.6, "secondary": 2.0, "tertiary": 1.4}
TIP_CSS = ("font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;"
           "background:white;padding:9px 12px;border:1px solid #999;border-radius:6px;")
BOX = ("position:fixed;bottom:24px;left:24px;z-index:9999;background:white;"
       "padding:9px 12px;border:1px solid #999;border-radius:6px;"
       "font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;")
PANEL = ("position:fixed;top:20px;right:20px;z-index:9999;background:white;"
         "padding:9px 12px;border:1px solid #999;border-radius:6px;"
         "font-family:sans-serif;font-size:13px;line-height:1.5;max-width:265px;")


def swatches():
    """The legend rows, drawn as the road notebook draws them."""
    return "".join(
        f'<div><span style="display:inline-block;width:16px;height:4px;background:{c};'
        f'margin:0 6px 3px 0;vertical-align:middle;"></span>{label}</div>'
        for _, _, c, label in VOC_BANDS)


def band_colour(voc):
    for lo, hi, colour, _ in VOC_BANDS:
        if lo <= voc < hi:
            return colour
    return VOC_BANDS[-1][2]


with model.db_connection_spatial as _conn:
    _geo = pd.read_sql("SELECT link_id, AsBinary(geometry) wkb FROM links", _conn)
_geo["geometry"] = [shapely.wkb.loads(bytes(b)) for b in _geo["wkb"]]
_geo = gpd.GeoDataFrame(_geo.drop(columns="wkb"), geometry="geometry", crs="EPSG:4326")
_geo["geometry"] = _geo.to_crs(METRE_CRS).simplify(8).to_crs("EPSG:4326")
_geo["geometry"] = shapely.set_precision(_geo.geometry.values, 1e-5)


def map_periods(filename="0_time_of_day.html"):
    """Every period as its own layer, so switching between them shows the peak turn round."""
    centre = zones.to_crs("EPSG:4326").geometry.union_all().centroid
    # The basemap is added separately with control=False. As base layers the hours share a radio
    # group, and a tile layer left in the control would sit in that group and be switched off
    # whenever an hour is chosen.
    m = folium.Map(location=[centre.y, centre.x], zoom_start=11,
                   tiles=None, prefer_canvas=True)
    folium.TileLayer("CartoDB positron", control=False).add_to(m)
    for period in PERIODS:
        f = loaded_streets(results[period])
        drawn = f[f["link_type"].isin(MAP_ROAD_TYPES)].merge(_geo, on="link_id", how="inner")
        drawn = gpd.GeoDataFrame(drawn, geometry="geometry", crs="EPSG:4326")
        # One GeoJson per period rather than a PolyLine per link: the geometry is written once
        # in compact arrays and the tooltip reads columns, instead of every link carrying its own
        # rendered HTML. The same map built link by link came to 72 MB.
        base = drawn["link_type"].str.replace("_link", "", regex=False)
        drawn["colour"] = [band_colour(v) for v in drawn["voc"]]
        drawn["weight"] = (base.map(LINE_WEIGHT).fillna(1.4)
                           * np.where(drawn["link_type"].str.endswith("_link"), 0.6, 1.0))
        drawn["street"] = drawn["name"].fillna("unnamed").replace("", "unnamed")
        drawn["A to B"] = drawn[CAR_AB].fillna(0).round(0)
        drawn["B to A"] = drawn[CAR_BA].fillna(0).round(0)
        drawn["V/C"] = drawn["voc"].round(2)
        fields = ["street", "A to B", "B to A", "V/C", "link_id"]
        folium.GeoJson(
            drawn[fields + ["colour", "weight", "geometry"]],
            name=f"{period} ({HOURS[period]})",
            overlay=False,                      # a base layer: picking one drops the rest
            show=period == "morning peak",
            style_function=lambda feature: {
                "color": feature["properties"]["colour"],
                "weight": feature["properties"]["weight"],
                "opacity": 0.85},
            highlight_function=lambda _feature: {"weight": 6, "opacity": 1},
            tooltip=folium.GeoJsonTooltip(
                fields=fields,
                aliases=["", "A&rarr;B veh/h", "B&rarr;A veh/h", "V/C", "link"],
                sticky=False, style=TIP_CSS),
        ).add_to(m)
        print(f"  {period:<15}{len(drawn):>7,} links drawn")

    m.get_root().html.add_child(folium.Element(
        f'<div style="{BOX}"><b>Volume over capacity</b>{swatches()}</div>'))
    folium.LayerControl(collapsed=False).add_to(m)
    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"\nMap saved: {filename}  ({os.path.getsize(path) / 1e6:.1f} MB)")
    return m


map_periods()

### One map per hour

The switcher holds four copies of the network, so it leaves out tertiary streets. These do not:
same classes as the road notebook's congestion map, and every link attribute on the hover.

In [ ]:
DETAIL_ROAD_TYPES = MAP_ROAD_TYPES + ["tertiary", "tertiary_link"]
DETAIL_FILES = {"morning peak": "1_morning_peak.html", "midday": "2_midday.html",
                "evening peak": "3_evening_peak.html", "night": "4_night.html"}


def map_hour_detail(period, filename):
    """One hour, every through-road, and the numbers behind each link."""
    f = loaded_streets(results[period])
    drawn = f[f["link_type"].isin(DETAIL_ROAD_TYPES)].merge(_geo, on="link_id", how="inner")
    drawn = gpd.GeoDataFrame(drawn, geometry="geometry", crs="EPSG:4326")

    base = drawn["link_type"].str.replace("_link", "", regex=False)
    drawn["colour"] = [band_colour(v) for v in drawn["voc"]]
    drawn["weight"] = (base.map(LINE_WEIGHT).fillna(1.4)
                       * np.where(drawn["link_type"].str.endswith("_link"), 0.6, 1.0))

    # What the model assumed about the link, and what it did with it.
    drawn["street"] = drawn["name"].fillna("unnamed").replace("", "unnamed")
    drawn["class"] = drawn["link_type"]
    drawn["lanes"] = drawn["lanes_ab"].fillna(0).astype(int)
    drawn["free-flow km/h"] = drawn["speed_ab"].fillna(0).round(0)
    drawn["capacity veh/h"] = drawn["capacity_ab"].fillna(0).round(0)
    drawn["A to B veh/h"] = drawn[CAR_AB].fillna(0).round(0)
    drawn["B to A veh/h"] = drawn[CAR_BA].fillna(0).round(0)
    drawn["V/C A to B"] = drawn["voc_ab"].round(2)
    drawn["V/C B to A"] = drawn["voc_ba"].round(2)
    # How much slower than an empty street, on the busier side.
    _busy_free = np.where(drawn[CAR_AB].fillna(0) >= drawn[CAR_BA].fillna(0),
                          drawn["travel_time_ab"], drawn["travel_time_ba"])
    _busy_now = np.where(drawn[CAR_AB].fillna(0) >= drawn[CAR_BA].fillna(0),
                         drawn["Congested_Time_AB"], drawn["Congested_Time_BA"])
    drawn["slower by"] = np.where(np.asarray(_busy_free, dtype=float) > 0,
                                  np.asarray(_busy_now, dtype=float)
                                  / np.maximum(np.asarray(_busy_free, dtype=float), 1e-9),
                                  np.nan).round(2)
    drawn["metres"] = drawn["distance"].round(0)

    fields = ["street", "class", "lanes", "free-flow km/h", "capacity veh/h", "metres",
              "A to B veh/h", "B to A veh/h", "V/C A to B", "V/C B to A", "slower by", "link_id"]
    centre = zones.to_crs("EPSG:4326").geometry.union_all().centroid
    m = folium.Map(location=[centre.y, centre.x], zoom_start=11, tiles=None, prefer_canvas=True)
    folium.TileLayer("CartoDB positron", control=False).add_to(m)
    folium.GeoJson(
        drawn[fields + ["colour", "weight", "geometry"]],
        name=period,
        style_function=lambda feature: {"color": feature["properties"]["colour"],
                                        "weight": feature["properties"]["weight"],
                                        "opacity": 0.85},
        highlight_function=lambda _feature: {"weight": 7, "opacity": 1},
        tooltip=folium.GeoJsonTooltip(
            fields=fields,
            aliases=["", "class", "lanes", "free-flow km/h", "capacity veh/h", "metres",
                     "A&rarr;B veh/h", "B&rarr;A veh/h", "V/C A&rarr;B", "V/C B&rarr;A",
                     "times free-flow", "link"],
            sticky=False, style=TIP_CSS),
    ).add_to(m)

    m.get_root().html.add_child(folium.Element(
        f'<div style="{BOX}"><b>Volume over capacity</b>{swatches()}</div>'))

    row = table.loc[period]
    over_km = drawn[drawn["voc"] >= 1.0]["distance"].sum() / 1000
    all_km = drawn["distance"].sum() / 1000
    m.get_root().html.add_child(folium.Element(
        f'<div style="{PANEL}"><b>{period}, {HOURS[period]}</b>'
        f'<div>{row["trips"]:,.0f} vehicle trips over {len(zones):,} zones</div>'
        f'<div>{row["veh-hours"]:,.0f} vehicle-hours, {row["delay %"]:.0f}% more '
        f'than the same trips on empty streets</div>'
        f'<div>{over_km:,.0f} of {all_km:,.0f} km drawn are over capacity</div>'
        f'</div>'))

    path = os.path.join(MAPS_DIR, filename)
    m.save(path)
    print(f"  {period:<15}{len(drawn):>7,} links   {filename:<22}"
          f"{os.path.getsize(path) / 1e6:>6.1f} MB")
    return m


print("detailed maps, one per hour:")
for _p in PERIODS:
    map_hour_detail(_p, DETAIL_FILES[_p])

## B4. What is assumed

Network, zones and matrix come from the road notebook unchanged. This notebook adds two numbers
of its own, neither measured:

| | |
|---|---|
| `PERIOD_SCALE` | midday 0.48 and night 0.17 against the morning peak; evening equals morning |
| `BPR_ALPHA, BPR_BETA` | 0.15 and 4.0, BPR's published values |

### What this does not capture

- **The hours do not interact.** Four static equilibria: no queue carries over, nobody shifts
  departure time. Both need dynamic assignment.
- **The hours do not add up to a day.** That needs the hours in each period, not modelled here.
- **`PERIOD_SCALE` is the whole temporal model.** Three unmeasured numbers. One count station
  with an hourly profile would replace them, and the AMT holds that data.
- **The evening is the morning reversed exactly.** The pure flip is the extreme of the standard
  blend, so no counter-flow exists in either peak. Real evening travel also carries more
  shopping and chained trips, and spreads over more hours.
- **One matrix stands in for every trip purpose.** Practice splits home-based work, home-based
  other and non-home-based, each with its own hourly profile, direction split and trip length.
  Here every hour shares one spatial pattern and one 16-minute average trip.
- **Capacity does not vary by hour.** Kerbside parking, loading and bus lanes all do.